# ImageCLEF 2026 – Deepfake Detection v2
### 실행 전: 런타임 > 런타임 유형 변경 > **T4 GPU** 선택

| 섹션 | 내용 | 예상 시간 |
|------|------|-----------|
| 0 | 환경 설정 | 3분 |
| 1 | 데이터 압축 해제 | 10~20분 |
| 2 | 이미지 딥페이크 탐지 | 20~30분 |
| 3 | 오디오 딥페이크 탐지 (wav2vec2 + 실제 데이터 기반) | 50~70분 |
| 4 | 제출 파일 생성 | 1분 |

## 0. 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q transformers torchaudio librosa scikit-learn pillow pandas tqdm accelerate
import torch
print('GPU:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU 이름:', torch.cuda.get_device_name(0))

## 1. 데이터 압축 해제

In [ ]:
import zipfile, os, glob

# ↓ Drive에 올린 zip 파일 경로 (필요시 수정)
TEST_ZIP = '/content/drive/MyDrive/ImageCLEF2026-DeepFakeDetection-Tes.zip'
REAL_ZIP = '/content/drive/MyDrive/Real_Data_Generation_Task.zip'

TEST_DIR = '/content/test_data'
REAL_DIR = '/content/real_data'

def extract(zip_path, out_dir):
    if not os.path.exists(out_dir):
        print(f'압축 해제 중: {os.path.basename(zip_path)} ...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(out_dir)
        print('완료!')
    else:
        print(f'이미 해제됨: {out_dir}')

extract(TEST_ZIP, TEST_DIR)
extract(REAL_ZIP, REAL_DIR)

In [ ]:
# 경로 자동 탐색
def find_files(base, ext):
    return glob.glob(os.path.join(base, '**', f'*.{ext}'), recursive=True)

# 테스트 데이터 경로
test_pngs = find_files(TEST_DIR, 'png')
test_wavs = find_files(TEST_DIR, 'wav')
IMG_CSV   = glob.glob(os.path.join(TEST_DIR, '**', 'Images_Detection_submission.csv'), recursive=True)[0]
AUD_CSV   = glob.glob(os.path.join(TEST_DIR, '**', 'Audio_Detection_submission.csv'),  recursive=True)[0]
IMAGE_DIR = os.path.dirname(test_pngs[0])
AUDIO_DIR = os.path.dirname(test_wavs[0])

# 실제(Real) 참조 데이터
real_wavs = find_files(REAL_DIR, 'wav')
real_pngs = find_files(REAL_DIR, 'png') + find_files(REAL_DIR, 'jpg') + find_files(REAL_DIR, 'jpeg')

print(f'[테스트] 이미지: {len(test_pngs)}개 | 오디오: {len(test_wavs)}개')
print(f'[실제]   오디오: {len(real_wavs)}개 | 이미지: {len(real_pngs)}개')
print(f'IMG CSV: {IMG_CSV}')
print(f'AUD CSV: {AUD_CSV}')

## 2. 이미지 딥페이크 탐지
> 모델: `dima806/deepfake_vs_real_image_detection` (ViT 기반)

In [ ]:
from transformers import pipeline
from PIL import Image
import pandas as pd
from tqdm.auto import tqdm
import os

device = 0 if torch.cuda.is_available() else -1

print('이미지 모델 로드 중...')
img_pipe = pipeline(
    'image-classification',
    model='dima806/deepfake_vs_real_image_detection',
    device=device
)
print('완료!')

In [ ]:
img_df    = pd.read_csv(IMG_CSV)
filenames = img_df['full_secret_name'].tolist()
print(f'총 이미지: {len(filenames)}개')

predictions = []
BATCH_SIZE  = 32

for i in tqdm(range(0, len(filenames), BATCH_SIZE), desc='이미지 추론'):
    batch_names  = filenames[i:i+BATCH_SIZE]
    batch_images = []
    valid_idx    = []

    for j, fname in enumerate(batch_names):
        try:
            img = Image.open(os.path.join(IMAGE_DIR, fname)).convert('RGB')
            batch_images.append(img)
            valid_idx.append(j)
        except:
            pass

    results  = img_pipe(batch_images)
    res_iter = iter(results)

    for j in range(len(batch_names)):
        if j in valid_idx:
            r     = next(res_iter)
            top   = r[0] if isinstance(r, list) else r
            label = top['label'].lower()
            predictions.append(1 if 'fake' in label or 'artificial' in label else 0)
        else:
            predictions.append(0)

img_df['prediction'] = predictions
print('\n예측 완료!')
print(img_df['prediction'].value_counts())

In [ ]:
import shutil
img_out = '/content/Images_Detection_submission.csv'
img_df.to_csv(img_out, index=False)
shutil.copy(img_out, '/content/drive/MyDrive/Images_Detection_submission.csv')
print('이미지 CSV 저장 완료!')
img_df.head(3)

## 3. 오디오 딥페이크 탐지
> wav2vec2 특징 추출 + 실제 데이터 기반 이상치 탐지 (반지도학습)

In [ ]:
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor
import torchaudio
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('wav2vec2 로드 중...')
feat_extractor = Wav2Vec2FeatureExtractor.from_pretrained('facebook/wav2vec2-base')
w2v = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base').to(device)
w2v.eval()
print('완료!')

In [ ]:
TARGET_SR   = 16000
MAX_SAMPLES = 16000 * 5  # 최대 5초
BATCH_SIZE  = 8

def load_wav(fpath):
    wav, sr = torchaudio.load(fpath)
    if sr != TARGET_SR:
        wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
    wav = wav.mean(dim=0)
    return wav[:MAX_SAMPLES].numpy()

def extract_w2v_features(file_list, desc='특징 추출'):
    all_feats  = []
    valid_idxs = []

    with torch.no_grad():
        for i in tqdm(range(0, len(file_list), BATCH_SIZE), desc=desc):
            batch_wavs = []
            batch_idx  = []

            for j, fpath in enumerate(file_list[i:i+BATCH_SIZE]):
                try:
                    wav = load_wav(fpath)
                    batch_wavs.append(wav)
                    batch_idx.append(i + j)
                except:
                    pass

            if not batch_wavs:
                continue

            inputs = feat_extractor(
                batch_wavs,
                sampling_rate=TARGET_SR,
                return_tensors='pt',
                padding=True
            ).input_values.to(device)

            # (B, T, 768) → mean pooling → (B, 768)
            feats = w2v(inputs).last_hidden_state.mean(dim=1).cpu().numpy()

            for k, vi in enumerate(batch_idx):
                all_feats.append(feats[k])
                valid_idxs.append(vi)

    return np.array(all_feats), valid_idxs

In [ ]:
# 실제(Real) 오디오 특징 추출
print(f'실제 오디오 {len(real_wavs)}개에서 특징 추출 중...')
real_feats, _ = extract_w2v_features(real_wavs, desc='실제 데이터 추출')
print(f'실제 특징: {real_feats.shape}')

In [ ]:
# 테스트 오디오 특징 추출
aud_df    = pd.read_csv(AUD_CSV)
filenames = aud_df['full_secret_name'].tolist()
test_paths = [os.path.join(AUDIO_DIR, f) for f in filenames]

print(f'테스트 오디오 {len(test_paths)}개에서 특징 추출 중...')
test_feats, valid_idxs = extract_w2v_features(test_paths, desc='테스트 데이터 추출')
print(f'테스트 특징: {test_feats.shape}')

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import numpy as np

# 스케일링 (실제 데이터 기준으로 fit)
scaler     = StandardScaler()
real_scaled = scaler.fit_transform(real_feats)
test_scaled = scaler.transform(test_feats)

# KNN: 테스트 샘플이 실제 데이터 분포에서 얼마나 떨어져 있는지 측정
print('KNN 학습 중...')
knn = NearestNeighbors(n_neighbors=5, metric='cosine', n_jobs=-1)
knn.fit(real_scaled)

# 실제 데이터 자기 거리 (임계값 보정용)
real_self_dists, _ = knn.kneighbors(real_scaled)
real_mean = real_self_dists.mean(axis=1)
threshold  = real_mean.mean() + 2.0 * real_mean.std()
print(f'임계값: {threshold:.4f}  (실제 평균거리: {real_mean.mean():.4f} ± {real_mean.std():.4f})')

# 테스트 거리 계산
test_dists, _ = knn.kneighbors(test_scaled)
test_mean_dists = test_dists.mean(axis=1)

# 분류: 실제 분포에서 멀수록 → Deepfake
preds = np.zeros(len(filenames), dtype=int)
for i, vi in enumerate(valid_idxs):
    preds[vi] = 1 if test_mean_dists[i] > threshold else 0

aud_df['prediction'] = preds
print('\n예측 완료!')
print(aud_df['prediction'].value_counts())

In [ ]:
aud_out = '/content/Audio_Detection_submission.csv'
aud_df.to_csv(aud_out, index=False)
shutil.copy(aud_out, '/content/drive/MyDrive/Audio_Detection_submission.csv')
print('오디오 CSV 저장 완료!')
aud_df.head(3)

## 4. 최종 제출 파일 생성

In [ ]:
import zipfile, shutil

submit_zip = '/content/submission.zip'
with zipfile.ZipFile(submit_zip, 'w') as z:
    z.write('/content/Images_Detection_submission.csv', 'Images_Detection_submission.csv')
    z.write('/content/Audio_Detection_submission.csv',  'Audio_Detection_submission.csv')

shutil.copy(submit_zip, '/content/drive/MyDrive/submission.zip')

# 최종 확인
img_check = pd.read_csv('/content/Images_Detection_submission.csv')
aud_check = pd.read_csv('/content/Audio_Detection_submission.csv')
print(f'[이미지] {len(img_check)}개 | 빈값: {img_check["prediction"].isna().sum()}')
print(img_check['prediction'].value_counts())
print(f'\n[오디오] {len(aud_check)}개 | 빈값: {aud_check["prediction"].isna().sum()}')
print(aud_check['prediction'].value_counts())
print('\n제출 파일 생성 완료: submission.zip')

In [ ]:
# Colab에서 직접 다운로드
from google.colab import files
files.download('/content/submission.zip')